# 01 · Dataset and instance

**What it does.** Loads the QOPTLib vehicle routing instance P-n6_1 (`load_instance`): one depot, five
customers with 2-D coordinates and a vehicle capacity. It builds the distance matrix and recomputes the
published reference solutions with the project's `objective` function.

**Why it was implemented.** Every later QUBO uses these distances, so the conventions must be right. The
`.vrp` files do not state the depot, the demands or the number of vehicles. The project infers them
(depot = first node, demand 1 per customer, 2 vehicles, TSPLIB rounding of each distance to the nearest
integer) and this notebook checks them: if the conventions are right, every published reference cost is
reproduced exactly. P-n6_1 was chosen because it fits the 8–12-qubit scope after decomposition and its
reference solution is feasible and optimal (dataset criteria in `agent-identity.md`).

**Resources.**
- Data: `data/vrp/P-n6_1.vrp` and `data/vrp/Solutions VRP.txt` (QOPTLib VRP subset); verdict and conventions in `data/vrp/README.md`.
- Code: `domains/vrp/problem.py` (`load_instance`, `Solution`, `objective`, `is_feasible`).
- Libraries: Python standard library (`math.dist`), matplotlib for the map.
- Literature: E. Osaba, E. Villar-Rodriguez, *QOPTLib*, Springer 2023 (arXiv:2404.15852); G. Reinelt, *TSPLIB*, ORSA J. Computing 1991 (EUC_2D rounding).

In [ ]:
# Run from anywhere: move to the repository root so `common/`, `domains/`, `data/` resolve.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
from domains.vrp.problem import Solution, is_feasible, load_instance, objective

inst = load_instance("data/vrp/P-n6_1.vrp")
print(inst.name, "| capacity", inst.capacity, "| vehicles", inst.n_vehicles, "| customers", list(inst.customers))
for i, (x, y) in enumerate(inst.coords):
    print(f"node {i}: ({x:g}, {y:g})", "<- depot" if i == 0 else "")

## Distance matrix
`d[i][j] = int(sqrt(dx² + dy²) + 0.5)`: Euclidean distance rounded per edge, the TSPLIB `EUC_2D` rule.

In [ ]:
print("     " + "".join(f"{j:>5}" for j in range(len(inst.coords))))
for i, row in enumerate(inst.dist):
    print(f"{i:>5}" + "".join(f"{d:>5}" for d in row))

## The reference solution
QOPTLib lists routes 0-1-5-2-0 and 0-4-3-0 with cost 118 for P-n6_1. `objective` sums depot → customers in
the stored order → depot for each vehicle.

In [ ]:
reference = Solution(inst, ((1, 5, 2), (4, 3)))
print("feasible:", is_feasible(reference), "| total cost:", objective(reference))
for route in reference.routes:
    print(f"  0-{'-'.join(map(str, route))}-0: {objective(Solution(inst, (route,))):g}")

## Checking the conventions on all 10 QOPTLib instances
If depot, indexing and rounding are right, every published cost is reproduced. The capacity column also
shows which published references break the capacity (unit demand); those instances were not chosen.

In [ ]:
import ast
import re

lines = [l for l in Path("data/vrp/Solutions VRP.txt").read_text().splitlines() if l.strip()]
names = sorted((p.stem for p in Path("data/vrp").glob("*.vrp")),
               key=lambda s: (int(re.search(r"n(\d+)", s).group(1)), s))
matches = []
for name, line in zip(names, lines):
    routes_txt, ref = line.rsplit("-", 1)
    routes = tuple(tuple(edge[1] for edge in r[:-1]) for r in ast.literal_eval(routes_txt.strip()))
    sol = Solution(load_instance(f"data/vrp/{name}.vrp"), routes)
    matches.append(objective(sol) == int(ref))
    print(f"{name}: published {int(ref):>4} | recomputed {objective(sol):>5g} | loads {[len(r) for r in routes]}"
          f" | capacity {sol.instance.capacity} | within capacity: {is_feasible(sol)}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
for route, color, style in [((1, 5, 2), "#2a78d6", "-"), ((4, 3), "#eb6834", "--")]:
    path = [0, *route, 0]
    ax.plot([inst.coords[i][0] for i in path], [inst.coords[i][1] for i in path], style, color=color,
            linewidth=2, label=f"0-{'-'.join(map(str, route))}-0 (length {objective(Solution(inst, (route,))):g})")
for i, (x, y) in enumerate(inst.coords):
    ax.plot(x, y, "s" if i == 0 else "o", color="#14213D", markersize=10)
    ax.annotate("depot" if i == 0 else str(i), (x, y), textcoords="offset points", xytext=(8, 6))
ax.set_title("P-n6_1: optimal routes, total 118")
ax.set_aspect("equal")
ax.legend()
plt.show()

## Expected results
- Depot (42, 57); customers 1–5; capacity 3; 2 vehicles.
- Distance row of the depot: 0, 19, 10, 18, 34, 13.
- Reference solution feasible, cost **118** = 45 (0-1-5-2-0) + 73 (0-4-3-0).
- All 10 published costs reproduced. P-n7_1, P-n7_2 and P-n8_2 references exceed the capacity under unit demand.

In [ ]:
assert inst.dist[0] == (0, 19, 10, 18, 34, 13)
assert is_feasible(reference) and objective(reference) == 118
assert all(matches) and len(matches) == 10